# AllDap 채점 모델 비교 v2 (M3, M4)

설계: `docs/superpowers/specs/2026-10-05-judge-selfhost-v2-design.md`

**실행 방법**
1. 런타임 > 런타임 유형 변경 > **T4 GPU**
2. 왼쪽 열쇠 아이콘(비밀값)에 `HF_TOKEN` 을 넣는다(Hugging Face 토큰, 읽기 권한). 노트북 접근을 허용한다.
3. Google Drive 의 `MyDrive/alldap_judge5v2/` 폴더에 `cases.jsonl` 을 올린다.
4. 런타임 > **모두 실행**. Drive 연결 허용 창이 뜨면 허용한다.
5. 끊기면 같은 순서로 다시 **모두 실행** 한다. 이미 채점한 문항은 건너뛴다.
   **오류로 멈춘 뒤 다시 실행할 때는 런타임 > 세션 다시 시작 후 모두 실행** 한다. 같은 커널에서 이어 돌리면 앞 모델이 GPU 에 남아 다음 모델이 메모리 부족으로 잘못 건너뛰어진다(2026-10-02 에 겪었다).
6. 끝나면 마지막 칸이 `alldap_judge5v2_results.zip` 을 내려받는다. 중간에 오류로 멈췄으면 마지막 칸만 따로 실행해 그때까지의 결과를 받는다.

이번 실험은 시험지를 **한 번만** 채점한다(v2 스펙 5절). 앞 실험에서 두 번 채점한 결과가 M3, M4 모두 같았기 때문이다. M2 는 앞 실험에서 T4 메모리에 올라가지 않아 뺐다.

모델 하나가 메모리에 안 올라가면 그 모델은 건너뛰고 이유를 `{모델}_skipped.json` 에 남긴다. 다른 모델로 바꿔 끼우지 않는다. 건너뜀 기록이 있는 모델은 다시 실행해도 건너뛴다(탐욕적 디코딩이라 다시 해도 같은 곳에서 모자란다). 다시 시도하려면 그 파일을 지운다.

모델마다 시험 문항 전에 가짜 문항 하나로 먼저 확인하고 출력을 `{모델}_smoke.json` 에 남긴다. 출력이 비어 있으면 멈춘다. `[RESULT]` 가 없는 것만으로는 멈추지 않는다. 양식을 안 따르는 것도 그 모델의 성적이고, 보고서가 못 읽음으로 센다.

디스크: 원본 가중치가 M3, M4 각 약 30GB 라 모델 하나를 끝낼 때마다 그 모델의 내려받은 파일을 지운다.

In [ ]:
# transformers 는 로컬에서 토크나이저와 대화 틀을 확인한 판으로 고정한다. 판이 바뀌면 기본 정밀도나
# 대화 틀 처리가 달라질 수 있는데, 그 차이가 채점 결과에 섞이면 가를 수 없다.
!pip install -q "transformers==4.57.6" accelerate bitsandbytes huggingface_hub

In [ ]:
import gc, hashlib, json, os, platform, shutil, sys, traceback
import torch, transformers, bitsandbytes
from google.colab import drive, userdata, files
from huggingface_hub import hf_hub_download, model_info
from huggingface_hub.constants import HF_HUB_CACHE
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig

drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/alldap_judge5v2'
CASES = f'{DIR}/cases.jsonl'
OUT = f'{DIR}/results'
os.makedirs(OUT, exist_ok=True)
HF_TOKEN = userdata.get('HF_TOKEN')

EXPECTED_SHA256 = "bec2b58e943e3b2c296dec8b664dc78ed2548f1b6d6e67062e3a7684fd535fe2"
MAX_NEW_TOKENS = 512
REPOS = {
    "M3": "Unbabel/M-Prometheus-14B",
    "M4": "Qwen/Qwen2.5-14B-Instruct",
}
# 동작 확인용 가짜 사례(시험 문항과 무관). tools/judge5.py 의 smoke_messages() 를 글자 그대로 옮긴 것이고,
# tools.judge5v2_check 가 같은지 확인한다.
SMOKE_MESSAGES = [{"role": "system", "content": "You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance."}, {"role": "user", "content": "###Task Description:\nAn instruction (might include an Input inside it), a response to evaluate, and a score rubric representing a evaluation criteria are given.\n1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.\n2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.\n3. The output format should look as follows: \"Feedback: (write a feedback for criteria) [RESULT] (an integer number between 1 and 5)\"\n4. Please do not generate any other opening, closing, and explanations.\n\n###The instruction to evaluate:\n아래 근거만 사용해서 질문에 답하세요.\n\n<근거>\n[근거 1] (출처: 가짜.md)\n정규직 직원의 연차는 15일이다.\n</근거>\n\n<질문>\n연차는 며칠인가요?\n\n###Response to evaluate:\n연차는 15일이고, 회사 창립기념일에는 하루 더 쉽니다.\n\n###Score Rubrics:\n[충실성: 답변의 모든 주장이 근거에 실제로 적혀 있는가. 답변이 세상의 상식에 맞는지는 보지 않는다. 사실이더라도 근거에 없으면 점수를 낮춘다.]\nScore 1: 답변이 근거와 관계없는 내용을 지어냈거나, 근거와 반대되는 내용을 말한다\nScore 2: 답변의 주장 대부분이 근거에 없다\nScore 3: 근거에 있는 주장과 근거에 없는 주장이 비슷한 정도로 섞여 있다\nScore 4: 답변의 핵심 답은 근거와 일치한다. 다만 근거에 없는 사소한 주장이 하나 있다\nScore 5: 답변의 모든 주장이 근거에 적혀 있다\n\n###Feedback: "}]

sha = hashlib.sha256(open(CASES, 'rb').read()).hexdigest()
assert sha == EXPECTED_SHA256, f'시험지 해시가 다릅니다: {sha}. 올린 파일이 맞는지 확인하세요.'
CASE_ROWS = [json.loads(l) for l in open(CASES, encoding='utf-8') if l.strip()]
print(len(CASE_ROWS), '건, 해시 확인 완료')

env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
       'transformers': transformers.__version__, 'bitsandbytes': bitsandbytes.__version__,
       'python': platform.python_version()}
json.dump(env, open(f'{OUT}/env.json', 'w'), ensure_ascii=False, indent=2)
print(env)

In [ ]:
QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16)  # T4 는 bf16 을 지원하지 않는다

def load(repo, revision):
    # revision: model_info 로 적은 커밋과 실제로 내려받는 판을 같게 묶는다. 없으면 그 사이에 저장소가 바뀔 수 있다.
    # Mistral 토크나이저는 불러올 때 "정규식이 틀려 토큰화가 잘못된다, fix_mistral_regex=True 를 주라" 고
    # 경고한다(2026-10-02 로컬 확인). 한국어 입력이 잘못 쪼개지면 Mistral 계열의 결과가 오염되므로 켠다(v2 는 M2 를 돌리지 않지만 앞 노트북과 같게 둔다).
    kw = {'fix_mistral_regex': True} if repo.startswith('mistralai/') else {}
    tok = AutoTokenizer.from_pretrained(repo, revision=revision, token=HF_TOKEN, **kw)
    if tok.chat_template is None:
        # Mistral-Small-3.1 은 대화 틀이 tokenizer_config.json 이 아니라 chat_template.json 에만 있다.
        # 그 파일을 직접 읽어 토크나이저에 단다. 틀의 글자는 저장소의 것 그대로다(우리가 쓰지 않는다).
        # AutoProcessor 로 읽지 않는 이유: 이미지 처리기까지 불러 쓸데없는 의존성(PIL)이 생긴다.
        path = hf_hub_download(repo, 'chat_template.json', revision=revision, token=HF_TOKEN)
        tok.chat_template = json.load(open(path, encoding='utf-8'))['chat_template']
    # dtype=torch.float16: 4비트로 바뀌지 않는 부분(임베딩, lm_head, 정규화층)의 정밀도다. 정하지 않으면
    # 판에 따라 float32(Mistral 의 임베딩과 lm_head 만 약 5.4GB)나 bf16(T4 미지원)이 되어, "모델이 커서
    # 안 올라간다" 와 "우리 설정 탓에 안 올라간다" 가 같은 건너뜀으로 뭉개진다. 4비트 계산 정밀도와 맞춘다.
    opts = dict(revision=revision, quantization_config=QUANT, dtype=torch.float16, device_map='auto', token=HF_TOKEN)
    try:
        model = AutoModelForCausalLM.from_pretrained(repo, **opts)
    except ValueError as e:
        if 'dispatched on the CPU' in str(e):
            raise   # GPU 메모리가 모자라 일부가 CPU 로 밀린 것이다. 아래 run_model 이 건너뜀으로 처리한다.
        # Mistral-Small-3.1 은 이미지도 받는 모델이라 CausalLM 으로 안 열린다. 글만 넣어 쓴다.
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(repo, **opts)
    model.eval()
    # 생성 설정을 저장소 기본값(generation_config.json) 위에 덮지 않고 처음부터 만든다.
    # Qwen2.5 와 M-Prometheus 의 기본값에는 repetition_penalty 1.05 가 있는데(Mistral 에는 없다), 탐욕적
    # 디코딩에서도 적용된다. 덮어쓰기 방식이면 빠뜨린 값이 M3, M4 에만 남아 세 모델의 조건이 달라진다(스펙 §3-3).
    # 저장소에서 가져오는 것은 멈춤 토큰(eos)뿐이다. 이것까지 버리면 Qwen 계열이 끝을 모르고 512 까지 쓴다.
    eos = model.generation_config.eos_token_id
    pad = tok.pad_token_id if tok.pad_token_id is not None else (eos[0] if isinstance(eos, list) else eos)
    gen = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0,
                           eos_token_id=eos, pad_token_id=pad)
    return tok, model, gen

def generate(tok, model, gen, messages):
    # 대화 틀로 글자를 먼저 만들고 토큰으로 바꾼다. apply_chat_template 에 return_tensors 를 주면
    # transformers 판에 따라 텐서 대신 dict 를 돌려줘서, 판과 무관한 이 두 단계로 나눴다.
    # add_special_tokens=False: 틀이 이미 시작 토큰(bos)을 넣으므로 한 번 더 붙이지 않는다.
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.no_grad():
        # use_model_defaults=False: generation_config 를 넘겨도 transformers 4.57 은 저장소 generation_config.json 의
        # 판이 4.50 이상이면 "전역 기본값과 같은 우리 값"(do_sample=False, repetition_penalty=1.0)을 저장소 값으로
        # 다시 덮는다. 지금 세 저장소는 판 숫자 덕에 안전하지만 그 안전을 판 숫자에 맡기지 않는다(리뷰에서 재현됨).
        out = model.generate(**enc, generation_config=gen, use_model_defaults=False)
    new = out[0, enc['input_ids'].shape[1]:]
    finish = 'length' if new.shape[0] >= MAX_NEW_TOKENS else 'stop'
    return tok.decode(new, skip_special_tokens=True), finish

def done_ids(path):
    if not os.path.exists(path):
        return set()
    return {json.loads(l)['case_id'] for l in open(path, encoding='utf-8') if l.strip()}

def gpu_gb():
    return round(torch.cuda.memory_allocated() / 1024**3, 2)

def skip_model(key, stage, case_id, reason, before_gb):
    # stage 는 load(올리기), smoke, run1 중 하나다(v2 는 한 번만 채점한다). "모델이 안 올라갔다" 와 "올라갔지만 근거가 긴
    # 사례 하나에서 모자랐다" 는 다른 사실이라 함께 적는다. 보고서는 실행 1 이 완전하면 그것을 쓴다.
    # before_gb: 이 모델을 올리기 전에 이미 차 있던 GPU 메모리다. 0 이 아니면 "모델이 커서 안 들어간다" 가
    # 아니라 "앞 모델이 남아 있었다" 일 수 있다. 두 사실이 같은 "메모리 부족" 으로 뭉개지지 않게 함께 적는다.
    # 보고서는 reason 만 읽으므로 reason 에도 넣고, 숫자로 다시 볼 수 있게 따로도 남긴다.
    after_gb = gpu_gb()
    reason = f'{reason} (올리기 전 GPU 점유 {before_gb}GB, 실패 시점 {after_gb}GB)'
    json.dump({'model': key, 'stage': stage, 'case_id': case_id, 'reason': reason,
               'gpu_allocated_before_load_gb': before_gb, 'gpu_allocated_at_failure_gb': after_gb},
              open(f'{OUT}/{key}_skipped.json', 'w'), ensure_ascii=False)
    print(key, '건너뜀:', stage, case_id, reason)

def run_model(key):
    repo = REPOS[key]
    skip = f'{OUT}/{key}_skipped.json'
    paths = [f'{OUT}/{key}_run{r}.jsonl' for r in (1,)]
    if os.path.exists(skip):
        print(key, '건너뜀 기록이 있어 다시 하지 않는다. 다시 하려면', skip, '를 지운다'); return
    if all(len(done_ids(p)) == len(CASE_ROWS) for p in paths):
        print(key, '이미 끝남'); return
    stage, case_id = 'load', None
    before_gb = gpu_gb()
    tok = model = None
    try:
        revision = model_info(repo, token=HF_TOKEN).sha
        tok, model, gen = load(repo, revision)
        stage = 'smoke'
        text, finish = generate(tok, model, gen, SMOKE_MESSAGES)
        # gen.to_dict(): 실제로 쓴 생성 설정을 남긴다. 저장소 값이 섞였는지 결과 문서에서 확인할 수 있게 한다.
        json.dump({'model': key, 'revision': revision, 'output': text, 'finish': finish, 'generation': gen.to_dict()},
                  open(f'{OUT}/{key}_smoke.json', 'w'), ensure_ascii=False, indent=2)
        print(key, 'smoke:', text[-200:])
        # 빈 출력만 멈춘다. [RESULT] 가 없는 것은 양식을 안 따른 것이고 그것도 이 모델의 성적이다(보고서가
        # missing 으로 센다). 여기서 멈추면 "양식 미준수" 가 "실행 불가" 로 둔갑해 모델이 표에서 빠진다.
        if not text.strip():
            raise RuntimeError(f'{key} 의 가짜 사례 출력이 비어 있다. {key}_smoke.json 을 보고 원인을 가린 뒤 진행한다.')
        for run, path in zip((1,), paths):
            stage = f'run{run}'
            done = done_ids(path)
            for i, row in enumerate(CASE_ROWS):
                if row['case_id'] in done:
                    continue
                case_id = row['case_id']
                text, finish = generate(tok, model, gen, row['messages'])
                line = {'case_id': row['case_id'], 'model': key, 'repo': repo, 'revision': revision,
                        'run': run, 'output': text, 'finish': finish, 'cases_sha256': sha,
                        'domain': row['domain']}
                with open(path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(line, ensure_ascii=False) + '\n')
                print(key, f'run{run}', i + 1, '/', len(CASE_ROWS), finish)
    except torch.cuda.OutOfMemoryError as e:
        skip_model(key, stage, case_id, f'GPU 메모리 부족: {e}', before_gb)
    except ValueError as e:
        if 'dispatched on the CPU' not in str(e):
            raise
        skip_model(key, stage, case_id, f'GPU 메모리 부족(4비트로도 다 올라가지 않아 일부가 CPU 로 밀림): {e}', before_gb)
    finally:
        # 예외로 나갈 때도 모델을 GPU 에서 내린다. IPython 은 마지막 예외의 traceback 을 들고 있고, traceback 은
        # 프레임을, 프레임은 지역 변수(model)를 붙잡는다. 그래서 함수가 끝나도 모델이 GPU 에 남아, 같은 커널에서
        # 다시 돌리면 다음 모델이 "메모리 부족" 으로 건너뛰어졌다(2026-10-02, M4 와 M2).
        # del 은 이 프레임의 이름을 지운다. clear_frames 는 traceback 이 붙잡은 더 안쪽 프레임(generate 의 model
        # 인자 등)의 지역 변수를 지운다. 오류 위치(줄 번호)는 남으므로 traceback 출력은 그대로 읽힌다.
        exc = sys.exc_info()[1]
        if exc is not None:
            traceback.clear_frames(exc.__traceback__)
        del tok, model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
for key in ('M3', 'M4'):
    try:
        run_model(key)
    except Exception:
        # 메모리 부족 말고 다른 이유로 실패하면 멈춘다. 다른 모델로 넘어가 결과를 섞지 않는다.
        traceback.print_exc()
        raise
    finally:
        # 모델을 GPU 에서 내리는 일은 run_model 의 finally 가 한다(예외로 나간 경우 포함).
        # 내려받은 가중치를 지운다. 두 모델을 합치면 약 60GB 라 코랩 디스크에 다 남길 수 없다.
        shutil.rmtree(os.path.join(HF_HUB_CACHE, 'models--' + REPOS[key].replace('/', '--')), ignore_errors=True)

In [ ]:
shutil.make_archive('/content/alldap_judge5v2_results', 'zip', OUT)
files.download('/content/alldap_judge5v2_results.zip')